# PDFPlumber Text Extraction — Evaluation Metrics

> These metrics evaluate the quality of text extracted by `pdfplumber` against manually curated reference files, covering extraction completeness, sequence preservation, noise, and overall similarity.

---

## Metrics at a Glance

| # | Metric | What It Measures |
|---|--------|-----------------|
| 1 | [Word Capture](#1-word-capture) | How much of the reference text was captured |
| 2 | [ROUGE-L](#2-rouge-l) | Sequence and structural similarity via LCS |
| 3 | [Word Precision](#3-word-precision) | How clean the extracted output is |
| 4 | [Word Recall](#4-word-recall) | How much reference content was recovered |
| 5 | [Word F1 Score](#5-word-f1-score) | Balanced precision/recall score |
| 6 | [Extracted Word Count](#6-extracted-word-count) | Total words in extracted text |
| 7 | [Reference Word Count](#7-reference-word-count) | Baseline total from reference text |
| 8 | [Missing Word Count](#8-missing-word-count) | Words in reference but absent from extraction |
| 9 | [Extra Word Count](#9-extra-word-count) | Words in extraction not in reference |

---

## 1. Word Capture

**Measures:** Extraction completeness — how much of the reference text was successfully captured.

$$
\text{Word Capture} = \frac{\text{Matched Reference Words}}{\text{Total Reference Words}}
$$

| Value | Meaning |
|-------|---------|
| `1.000` | All reference words were captured |
| `< 1.000` | Some content is missing |

**Why it matters:** Directly identifies whether information was lost during extraction.

---

## 2. ROUGE-L

**Measures:** Similarity between extracted and reference text using the Longest Common Subsequence (LCS).

$$
\text{ROUGE-L} = \frac{LCS(\text{Extracted}, \text{Reference})}{|\text{Reference}|}
$$

- Higher values → better preservation of text order and structure
- Sensitive to duplicated or reordered content

**Why it matters:** Evaluates reading order and sequence preservation, not just word presence.

---

## 3. Word Precision

**Measures:** The proportion of extracted words that are correct — i.e., extraction cleanliness.

$$
\text{Precision} = \frac{\text{Correct Words}}{\text{Correct Words} + \text{Extra Words}}
$$

- High precision → low noise, few spurious or duplicated words

**Why it matters:** Penalises over-extraction, duplicated content, and OCR artifacts.

---

## 4. Word Recall

**Measures:** The proportion of reference words successfully recovered from the extraction.

$$
\text{Recall} = \frac{\text{Correct Words}}{\text{Correct Words} + \text{Missing Words}}
$$

- High recall → fewer missing words, more complete extraction

**Why it matters:** Evaluates preservation of important content from the source document.

---

## 5. Word F1 Score

**Measures:** A balanced combination of precision and recall.

$$
F_1 = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

A high F1 score indicates both low missing content **and** low extraction noise simultaneously.

**Why it matters:** Provides a single overall quality score that doesn't reward trading one for the other.

---

## 6. Extracted Word Count

**Measures:** Total number of words in the extracted output.

Unusually large values may indicate:
- Duplicated words or phrases
- Repeated headers or footers
- Extraction noise from non-content regions

**Why it matters:** Useful for spotting over-extraction before diving into precision/recall.

---

## 7. Reference Word Count

**Measures:** Total number of words in the manually curated reference text.

**Why it matters:** Serves as the baseline denominator for all completeness and recall calculations.

---

## 8. Missing Word Count

**Measures:** Words present in the reference but absent from the extraction.

$$
\text{Missing Words} = \text{Reference Words} - \text{Extracted Words}
$$

Higher values indicate extraction failures or lost content regions.

**Why it matters:** Directly quantifies information loss.

---

## 9. Extra Word Count

**Measures:** Words present in the extraction that do not exist in the reference.

$$
\text{Extra Words} = \text{Extracted Words} - \text{Reference Words}
$$

Higher values may indicate:
- Duplicated content
- Extraction noise
- OCR artifacts or spurious characters

**Why it matters:** Directly quantifies extraction pollution and duplication errors.

---

## Summary

Together, these nine metrics provide a comprehensive view of PDF text extraction quality across five dimensions:

| Dimension | Relevant Metrics |
|-----------|-----------------|
| **Completeness** | Word Capture, Word Recall, Missing Word Count |
| **Cleanliness** | Word Precision, Extra Word Count |
| **Overall quality** | Word F1 Score |
| **Structural fidelity** | ROUGE-L |
| **Volume diagnostics** | Extracted Word Count, Reference Word Count |

> These metrics were developed to evaluate `pdfplumber` for extracting machine-readable text from Data Management Plan (DMP) PDFs within the **DMPBridge** project.


In [1]:
import sys
print("Python:", sys.executable)
print()
try:
    from dmpbridge.evaluation.pdfplumber_text_evaluator import evaluate_pdfplumber_text
    print("Import OK")
except Exception as e:
    print("Import FAILED:", type(e).__name__, e)

Python: C:\Users\Nahid\AppData\Local\Programs\Python\Python310\python.exe

Import OK


## Part 1 — Imports


In [2]:
import pandas as pd
from pathlib import Path

from dmpbridge.evaluation.pdfplumber_text_evaluator import evaluate_pdfplumber_text

## Part 2 — Project paths


In [3]:
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

extracted_folder = project_root / "data" / "pdfplumber_extracted_text"
reference_folder = project_root / "data" / "reference_text"

print("Project root:", project_root)
print("Extracted text folder exists:", extracted_folder.exists())
print("Reference folder exists:", reference_folder.exists())

Project root: C:\Users\Nahid\dmpbridge
Extracted text folder exists: True
Reference folder exists: False


## Part 3 — Run evaluation on all PDFs


In [4]:
results = []

for extracted_file in sorted(extracted_folder.glob("*.txt")):

    if extracted_file.stem.endswith("_cleaned"):
        continue

    sample_id = extracted_file.stem
    reference_file = reference_folder / f"{sample_id}_reference.txt"

    if not reference_file.exists():
        print(f"Missing reference file: {reference_file}")
        continue

    result = evaluate_pdfplumber_text(
        extracted_txt_path=extracted_file,
        reference_txt_path=reference_file,
        clean_text=True,
    )

    results.append(result)

df = pd.DataFrame(results)

display(df)

Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample1_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample10_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample2_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample3_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample4_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample5_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample6_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample7_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample8_reference.txt
Missing reference file: C:\Users\Nahid\dmpbridge\data\reference_text\sample9_reference.txt


""
